# Strict ten-session features: coverage and lookahead inspection

Exactly ten completed exchange sessions, nine internal overnight gaps, and no pre-window or trade-day price. The oldest session has no gap-dependent columns. This produces 267 declared inputs. No model is trained. All source rows with resolved nonconflicting signal-day membership remain; missing features never silently remove a row.

Risk and relative-activity references are computed at the end of this same completed block. They may use sessions later than an older historical day, but are all known at the prediction cutoff. This is not an as-of-opening feature reconstruction for those older days. Older EMA/MACD state and the old forecast eligibility gate are excluded.

In [ ]:
from pathlib import Path
import sys, json
import numpy as np
import pandas as pd
import duckdb
from IPython.display import display, Image
PROJECT_DIR = Path.cwd()
if not (PROJECT_DIR / "data").exists():
    PROJECT_DIR = PROJECT_DIR.parent
sys.path.insert(0, str(PROJECT_DIR / "scripts"))
RELEASE = PROJECT_DIR / "data/features/15_session_window_10_v1"
REPORT = PROJECT_DIR / "data/reports/15_build_session_window_features"
BUILD_IF_MISSING = False
if not (RELEASE / "MANIFEST.json").exists():
    if BUILD_IF_MISSING:
        from build_session_window_features import build
        build(PROJECT_DIR)
    else:
        raise FileNotFoundError("Build the declared feature release explicitly first; inspection does not rebuild it.")
manifest = json.loads((RELEASE / "MANIFEST.json").read_text())
assert manifest["complete"] and manifest["model_fits"] == 0
assert manifest["window_sessions"] == 10 and manifest["internal_overnight_gaps"] == 9
assert not manifest["current_trade_day_inputs"] and not manifest["older_smoothing_state"]
display(pd.Series(manifest["summary"], name="count").to_frame())
print(manifest["cohort"])
print(manifest["normalization"])

In [ ]:
quality = pd.read_parquet(RELEASE / "coverage_ledger.parquet")
display(quality.groupby(quality.signal_date.dt.year).agg(
    rows=("security_id", "size"), price_complete=("window_price_complete", "mean"),
    activity_complete=("window_activity_complete", "mean"), all_features_finite=("complete_feature_vector", "mean"),
    mean_missing_features=("missing_features", "mean")))
print("Rows with all 267 inputs missing:", int(quality.missing_features.eq(267).sum()))
reason_names = {int(k): v for k, v in json.loads((RELEASE / "MISSING_REASONS.json").read_text()).items()}
reason_counts = np.zeros(max(reason_names) + 1, dtype=np.int64)
for start in range(0, len(quality), 30000):
    values = np.frombuffer(b"".join(quality.missing_reason_bytes.iloc[start:start+30000]), dtype=np.uint8)
    reason_counts += np.bincount(values, minlength=len(reason_counts))
display(pd.DataFrame({"reason": [reason_names[k] for k in range(len(reason_counts))], "feature_cells": reason_counts}))
display(pd.read_csv(RELEASE / "feature_coverage.csv"))
display(Image(filename=str(RELEASE / "coverage.png")))
print("Complete-case counts are descriptive, not an eligibility rule. Flat openings/zero gaps can have valid prices and undefined features.")

In [ ]:
features = manifest["model_features"]
display(pd.read_csv(RELEASE / "feature_dictionary.csv"))
duplicates = json.loads((RELEASE / "DUPLICATE_AUDIT.json").read_text())
display({key: duplicates[key] for key in ("duplicate_groups", "constants", "all_missing")})
assert len(features) == len(set(features)) == 267
assert len([f for f in features if f.endswith("_gap")]) == 9
assert not any(f.startswith("lag10_gap") for f in features)
assert manifest["construction_checks_passed"]
audit_path = REPORT / "WINDOW_AUDIT.json"
if not audit_path.exists():
    raise FileNotFoundError("Run scripts/audit_session_window_features.py before interpreting construction results.")
audit = json.loads(audit_path.read_text())
assert audit["all_passed"]
display(pd.DataFrame(audit["checks"]))
display(pd.read_csv(REPORT / "actual_data_audit_samples.csv"))

In [ ]:
con = duckdb.connect()
con.execute("SET threads=4")
columns = ["security_id", "signal_date", "trade_date", "lag01_open_05m", "lag01_open_30m",
    "lag01_follow_30m", "lag01_persist_30m", "lag01_gap", "lag09_gap", "lag01_activity_05m", "lag10_realized_vol"]
sample = con.execute("SELECT " + ",".join(columns) + " FROM read_parquet(?) WHERE signal_date=DATE '2024-03-22' ORDER BY security_id LIMIT 10", [str(RELEASE / "features.parquet")]).df()
display(sample)
# One complete price/activity window, displayed as a chronological table.
complete = quality[quality.signal_date.eq(pd.Timestamp("2024-03-22")) & quality.window_price_complete & quality.window_activity_complete].iloc[0]
example = con.execute("SELECT * FROM read_parquet(?) WHERE security_id=? AND signal_date=DATE '2024-03-22'", [str(RELEASE / "features.parquet"), complete.security_id]).df().iloc[0]
descriptors = ["open_05m", "open_30m", "follow_30m", "persist_30m", "gap", "gap_filled", "realized_vol", "activity_05m"]
view = pd.DataFrame([{name: example.get(f"lag{lag:02}_{name}", np.nan) for name in descriptors} for lag in range(10, 0, -1)], index=[f"lag{lag:02}" for lag in range(10, 0, -1)])
display(view)
print("The oldest displayed gap cells are intentionally absent from the model schema.")
selected = quality[quality.signal_date.eq(pd.Timestamp("2024-03-22"))].iloc[0]
reasons = {int(k): v for k, v in json.loads((RELEASE / "MISSING_REASONS.json").read_text()).items()}
codes = np.frombuffer(selected.missing_reason_bytes, dtype=np.uint8)
display(pd.DataFrame({"feature": features, "reason": [reasons[int(k)] for k in codes]}).loc[codes != 0])
legacy = pd.read_parquet(RELEASE / "legacy_cohort_coverage.parquet")
display(legacy[["window_price_complete", "window_activity_complete", "complete_feature_vector"]].mean().rename("legacy_cohort_coverage").to_frame())
con.close()

No model fitting or training-pool selection occurs here. The broader resolved-member panel and the former model cohort are reported separately. Membership and adjustment vintages remain retrospective; numerical checks do not certify historical PIT or tradability. Present five-minute bars with 1–4 underlying source minutes are admitted and flagged; no one-minute files are read.